# Skill with Additional Python Resource

This notebook demonstrates using a skill bundle that includes a Python script resource. The `word-frequency` skill bundles:

- `scripts/word_freq.py` — a script that reads text from stdin and prints the top-10 most frequent words as a markdown table

The skill instructs the agent to read and then execute the script using `ReadFileTool` and `PythonInterpreterTool`, passing the user's text as `stdin`. The user supplies any text passage directly in their task instruction. Word counting is a task LLMs get wrong on raw text — the script guarantees deterministic, accurate results.

The skill bundle is at `.agents/skills/word-frequency/` and is discovered automatically when running from `more-examples/ch06/`.

**Note:** Run this notebook from within the `more-examples/ch06/` directory so the `word-frequency` skill is discovered as a project-scoped skill.

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPI by uncommenting the cell below.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

## Inspecting the skill bundle

Before running the agent, let's verify the skill is discovered and that `Skill.resources` picks up `scripts/word_freq.py`.

In [7]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM

llm = OllamaLLM(model="qwen3:14b", think=False)
agent = LLMAgent(llm=llm)

task = Task(instruction="placeholder")
handler = LLMAgent.TaskHandler(llm_agent=agent, task=task)
skill = handler.skills["word-frequency"]

print(f"Name:      {skill.frontmatter.name}")
print(f"Scope:     {skill.scope}")
print("Resources:")
for r in skill.resources:
    print(f"  - {r}")
print()
print("--- Body ---")
print(skill.read_body())

Name:      word-frequency
Scope:     SkillScope.PROJECT
Resources:
  - scripts/word_freq.py

--- Body ---
# Word Frequency

This skill counts word frequencies in a text passage provided by the user and reports the top-10 results as a markdown table. The computation is performed by a Python script to ensure deterministic, accurate counts.

## Steps

### 1. Read the script

Inspect the bundled script before running it:

```
from_scratch__read_file(path="<skill_dir>/scripts/word_freq.py")
```

Replace `<skill_dir>` with the value of **Skill directory** shown at the bottom of this skill content.

### 2. Execute the script

Run the script, passing the user's text as `stdin`:

```
from_scratch__python_interpreter(path="<skill_dir>/scripts/word_freq.py", stdin="<user_text>")
```

Replace `<user_text>` with the exact text passage the user provided.

### 3. Report the result

Report the markdown table printed by the script exactly as-is.


### Reading the additional resource

We can read the script content directly via the resource path.

In [8]:
script_resource = next(r for r in skill.resources if r.suffix == ".py")
script_path = skill.location.parent / script_resource
print(script_path.read_text())

"""Read text from stdin and print the top-10 most frequent words.

Output is a markdown table.
"""

import re
import sys
from collections import Counter

text = sys.stdin.read().lower()
words = re.findall(r"[a-z']+", text)
top10 = Counter(words).most_common(10)

print("**Top-10 word frequencies**\n")
print("| Rank | Word | Count |")
print("|------|------|-------|")
for rank, (word, count) in enumerate(top10, 1):
    print(f"| {rank} | {word} | {count} |")



## Running the agent

Now let's run the agent on a task that includes a text passage. The agent will discover the `word-frequency` skill, activate it via `from_scratch__use_skill`, then follow the skill's instructions to read `scripts/word_freq.py` and execute it with the user's text piped as `stdin`.

In [9]:
import logging

from llm_agents_from_scratch.logger import enable_console_logging

enable_console_logging(logging.INFO)

In [ ]:
from llm_agents_from_scratch import TOOLS_FOR_SKILL_RESOURCES

PASSAGE = """\
Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than right now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!
"""

agent_with_tools = LLMAgent(llm=llm, tools=TOOLS_FOR_SKILL_RESOURCES)
task = Task(
    instruction=(
        "Use the word-frequency skill to find the top-10 most frequent"
        f" words in the following passage:\n\n{PASSAGE}"
    ),
)
handler = agent_with_tools.run(task)

In [11]:
await handler
result = handler.exception() or handler.result()
print(result)

CancelledError: 

In [13]:
handler

<TaskHandler cancelled>